In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500, random_state=42) # random selection of rows
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
521,"LIGHTS OF NEW YORK was the first ""all-taking"" ...",positive
737,I think this is what this movie wants us to sa...,positive
740,So I'm looking to rent a DVD and I come across...,negative
660,Where was this film when I was a kid? After hi...,positive
411,I always enjoy seeing movies that make you thi...,positive


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
521,light new york first all taking feature film c...,positive
737,think movie want u say end movie damn australi...,positive
740,looking rent dvd come across movie called end ...,negative
660,film kid parent split tadashi move mom live gr...,positive
411,always enjoy seeing movie make think drip feed...,positive


In [5]:
# it gives the class distribution (or frequency distribution) of a feature
df['sentiment'].value_counts()

sentiment
positive    252
negative    248
Name: count, dtype: int64

In [6]:
# It filters the DataFrame to keep only rows where the 'sentiment' column contains either "positive" or "negative".
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
521,light new york first all taking feature film c...,1
737,think movie want u say end movie damn australi...,1
740,looking rent dvd come across movie called end ...,0
660,film kid parent split tadashi move mom live gr...,1
411,always enjoy seeing movie make think drip feed...,1


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [13]:
# createing features with the top most frequent words in word vocab created by bag-of-words
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/....mlflow')
dagshub.init(repo_owner='<name>', repo_name='end-to-end-MLOPS-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

Initialized MLflow to track repo "rajaD1995/end-to-end-MLOPS-project"

2026-07-26 15:13:22,048 - INFO - Initialized MLflow to track repo "rajaD1995/end-to-end-MLOPS-project"


Repository rajaD1995/end-to-end-MLOPS-project initialized!

2026-07-26 15:13:22,050 - INFO - Repository rajaD1995/end-to-end-MLOPS-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/6b7d0b65b49f44cf90f43efcfd285efd', creation_time=1785058680978, experiment_id='0', last_update_time=1785058680978, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [16]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues in gradient bosting to find minima of error

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")
        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")
        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-26 15:13:37,677 - INFO - Starting MLflow run...
2026-07-26 15:13:38,500 - INFO - Logging preprocessing parameters...
2026-07-26 15:13:39,772 - INFO - Initializing Logistic Regression model...
2026-07-26 15:13:39,773 - INFO - Fitting the model...
2026-07-26 15:13:39,792 - INFO - Model training complete.
2026-07-26 15:13:39,793 - INFO - Logging model parameters...
2026-07-26 15:13:40,206 - INFO - Making predictions...
2026-07-26 15:13:40,208 - INFO - Calculating evaluation metrics...
2026-07-26 15:13:40,219 - INFO - Logging evaluation metrics...
2026-07-26 15:13:41,682 - INFO - Saving and logging the model...
/opt/anaconda3/envs/atlas/lib/python3.10/site-packages/_distutils_hack/__init__.py:11: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditi